# Память CSR-N на тензоре Uber pickups (FROSTT)

Источник: http://frostt.io/tensors/uber-pickups/

In [1]:
!pip install -q sparse

import os, gzip, shutil, urllib.request
import numpy as np
import sparse
from csrn import CSRN

URL = 'https://s3.us-east-2.amazonaws.com/frostt/frostt_data/uber-pickups/uber.tns.gz'
gz, tns = 'uber.tns.gz', 'uber.tns'

if not os.path.exists(tns):
    urllib.request.urlretrieve(URL, gz)
    with gzip.open(gz, 'rb') as f_in, open(tns, 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)

raw = np.loadtxt(tns, dtype=np.float64)
N = raw.shape[1] - 1
coords = raw[:, :N].astype(np.int64).T - 1
values = raw[:, -1].astype(np.float64)
shape = tuple(int(coords[k].max()) + 1 for k in range(N))
nnz = values.shape[0]

print(f'N={N}, shape={shape}, nnz={nnz:,}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.9/151.9 kB 5.2 MB/s eta 0:00:00
N=4, shape=(183, 24, 1140, 1717), nnz=3,309,490


In [2]:
csrn = CSRN.from_coo(coords, values, shape)
pds_coo = sparse.COO(coords, values, shape=shape)
pds_gcxs = sparse.GCXS.from_coo(pds_coo)

In [3]:
def mb(b): return f'{b / (1024**2):.1f} MB'

mem_csrn = csrn.memory_bytes()
mem_coo = pds_coo.nbytes
mem_gcxs = pds_gcxs.nbytes

n_leaves = int(csrn.level_offsets[N]) - int(csrn.level_offsets[N-1])
avg_per_prefix = nnz / n_leaves

print(f'CSR-N : {mb(mem_csrn)}')
print(f'COO   : {mb(mem_coo)}   (vs CSR-N: {mem_coo/mem_csrn:.2f}x)')
print(f'GCXS  : {mb(mem_gcxs)}   (vs CSR-N: {mem_gcxs/mem_csrn:.2f}x)')
print(f'\nn_leaves/nnz = {n_leaves/nnz:.3f}   (в среднем {avg_per_prefix:.2f} ненулевых на префикс)')

CSR-N : 66.5 MB
COO   : 126.2 MB   (vs CSR-N: 1.90x)
GCXS  : 50.5 MB   (vs CSR-N: 0.76x)

n_leaves/nnz = 0.209   (в среднем 4.78 ненулевых на префикс)


In [4]:
print(f'idx           : {mb(csrn.idx.nbytes)}')
print(f'values        : {mb(csrn.values.nbytes)}')
print(f'indptr        : {mb(csrn.indptr.nbytes)}')
print(f'base          : {mb(csrn.base.nbytes)}')
print(f'level_offsets : {mb(csrn.level_offsets.nbytes)}')

print(f'\nCOO.coords vs CSR-N.idx : {mb(pds_coo.coords.nbytes)} vs {mb(csrn.idx.nbytes)}   ({pds_coo.coords.nbytes/csrn.idx.nbytes:.2f}x)')

idx           : 30.6 MB
values        : 25.2 MB
indptr        : 5.3 MB
base          : 5.3 MB
level_offsets : 0.0 MB

COO.coords vs CSR-N.idx : 101.0 MB vs 30.6 MB   (3.30x)
